# AI-Powered Systematic Review Screening with GPT-5

This notebook implements an automated systematic review screening workflow using OpenAI's GPT-5 reasoning models with structured outputs.

**Features:**
- Uses GPT-5 reasoning models with adjustable reasoning effort
- Loads PubMed abstracts from CSV
- Reads inclusion/exclusion criteria from text file
- Uses OpenAI's structured outputs (Pydantic models) for consistent decision-making
- Follows PRISMA guidelines for exclusion reasoning
- Implements rate limiting and error handling
- Saves progress incrementally
- Generates PRISMA-style reporting

**Key GPT-5 Capabilities:**
- **Reasoning Effort**: Control depth of analysis (minimal, low, medium, high)
- **Structured Outputs**: Guaranteed Pydantic model compliance
- **Developer Messages**: Enhanced prompt control
- **No Temperature**: Uses reasoning_effort instead for consistency

**Requirements:**
- OpenAI SDK >= 1.0.0
- GPT-5, GPT-5-mini, or GPT-5-nano models

## 1. Setup and Imports

In [57]:
# Upgrade OpenAI SDK and visualization packages to latest versions
# Uncomment and run this cell if you haven't upgraded yet:

# !pip install --upgrade openai pydantic plotly matplotlib seaborn scikit-learn kaleido

# Note: kaleido is required for exporting Plotly figures to PDF
# If PDF export fails, run: pip install kaleido --upgrade

# Verify installation
import openai
print(f"OpenAI SDK version: {openai.__version__}")
print(f"✓ Version check: {'COMPATIBLE' if openai.__version__ >= '1.0.0' else 'NEEDS UPGRADE'}")

# Expected: 1.0.0 or higher

OpenAI SDK version: 2.7.1
✓ Version check: COMPATIBLE


## 0. Installation and Upgrade

**IMPORTANT:** This notebook requires OpenAI SDK >= 1.0.0 for structured outputs support.

Run the following command to upgrade:

In [59]:
import pandas as pd
import numpy as np
import json
import logging
import time
import os
from datetime import datetime
from typing import Literal, Optional
from tqdm import tqdm
from pydantic import BaseModel, Field
from openai import OpenAI
from secret_keys import OPENAI_API_KEY

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("ai_screening.log"),
        logging.StreamHandler()
    ]
)

# Initialize OpenAI client (SDK >= 1.0.0)
client = OpenAI(api_key=OPENAI_API_KEY)

logging.info("Setup complete - OpenAI client initialized")

2026-01-23 14:43:18,671 - INFO - Setup complete - OpenAI client initialized


## 2. Define Structured Output Schema using Pydantic

Using Pydantic BaseModel for type-safe, validated structured outputs.
The OpenAI API will enforce this schema strictly.

In [60]:
class ScreeningDecision(BaseModel):
    """
    Structured output model for systematic review screening decisions.
    
    This schema is strictly enforced by OpenAI's structured outputs feature,
    ensuring consistent, parseable responses following PRISMA guidelines.
    """
    decision: Literal["include", "exclude", "uncertain"] = Field(
        description="Final screening decision based on inclusion/exclusion criteria"
    )
    
    confidence: Literal["high", "medium", "low"] = Field(
        description="Confidence level in the screening decision"
    )
    
    exclusion_reason: Optional[Literal[
        "wrong_population",
        "wrong_exposure", 
        "wrong_outcome",
        "wrong_study_design",
        "animal_study",
        "no_original_data",
        "language",
        "duplicate",
        "insufficient_information",
        "not_applicable"
    ]] = Field(
        default=None,
        description="PRISMA-aligned exclusion reason (null if included)"
    )
    
    reasoning: str = Field(
        description="Brief explanation of the decision (1-3 sentences)"
    )
    
    population_match: bool = Field(
        description="Does the study population match criteria?"
    )
    
    exposure_match: bool = Field(
        description="Does the study examine the correct exposure?"
    )
    
    outcome_match: bool = Field(
        description="Does the study examine the correct outcome?"
    )
    
    study_design_appropriate: bool = Field(
        description="Is the study design appropriate (original research not systematic review, etc.)?"
    )

logging.info("Pydantic schema defined for structured outputs")

2026-01-23 14:43:35,786 - INFO - Pydantic schema defined for structured outputs


## 3. Load Inclusion Criteria

In [61]:
def load_inclusion_criteria(filepath="inclusion_criteria.txt"):
    """
    Load inclusion/exclusion criteria from text file.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            criteria_text = f.read()
        logging.info(f"Loaded inclusion criteria from {filepath}")
        return criteria_text
    except FileNotFoundError:
        logging.error(f"Inclusion criteria file not found: {filepath}")
        raise

# Load criteria
INCLUSION_CRITERIA = load_inclusion_criteria()
print(f"Criteria loaded ({len(INCLUSION_CRITERIA)} characters)")
print("\nFirst 500 characters:")
print(INCLUSION_CRITERIA[:500] + "...")

2026-01-23 14:43:37,861 - INFO - Loaded inclusion criteria from inclusion_criteria.txt


Criteria loaded (1859 characters)

First 500 characters:
Eligibility criteria
Studies will be eligible for inclusion if they reported on the following:
• Population: Pregnant human populations.
• Exposure: Elevated or high ambient temperatures or heat or heat indices pre-conception or during pregnancy.
• Comparator: Pregnancy population exposed to lower levels of heat
• Outcome: HDP, including gestational hypertension, pre-eclampsia (de novo or superimposed; early or late-onset), and related complications such as eclampsia and HELLP syndrome.1 Of note...


## 4. Load PubMed Data

In [62]:
def load_pubmed_data(filepath="outputs/labeled_clusters_subset_2.csv"):
    """
    Load PubMed results CSV file.
    """
    try:
        df = pd.read_csv(filepath)
        logging.info(f"Loaded {len(df)} records from {filepath}")
        
        # Ensure required columns exist
        required_cols = ['PMID', 'Title', 'Abstract']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        # Clean data
        df['Abstract'] = df['Abstract'].fillna('')
        df['Title'] = df['Title'].fillna('')
        
        # Filter out records without abstracts
        df_with_abstracts = df[df['Abstract'].str.strip() != ''].copy()
        logging.info(f"Records with abstracts: {len(df_with_abstracts)}")
        
        return df_with_abstracts
    except FileNotFoundError:
        logging.error(f"PubMed data file not found: {filepath}")
        raise

# Load data
df_pubmed = load_pubmed_data()
print(f"\nLoaded {len(df_pubmed)} articles with abstracts")
print(f"Columns: {df_pubmed.columns.tolist()}")
df_pubmed.head()

2026-01-23 14:43:40,160 - INFO - Loaded 237 records from outputs/labeled_clusters_subset_2.csv
2026-01-23 14:43:40,162 - INFO - Records with abstracts: 237



Loaded 237 articles with abstracts
Columns: ['PMID', 'Item Type', 'Publication Year', 'Author', 'Title', 'Publication Title', 'ISBN', 'ISSN', 'DOI', 'Url', 'Abstract', 'Date', 'Date Added', 'Date Modified', 'Access Date', 'Pages', 'Num Pages', 'Issue', 'Volume', 'Number Of Volumes', 'Journal Abbreviation', 'Short Title', 'Series', 'Series Number', 'Series Text', 'Series Title', 'Publisher', 'Place', 'Language', 'Rights', 'Type', 'Archive', 'Archive Location', 'Library Catalog', 'Call Number', 'Extra', 'Notes', 'File Attachments', 'Link Attachments', 'Manual Tags', 'Automatic Tags', 'Editor', 'Series Editor', 'Translator', 'Contributor', 'Attorney Agent', 'Book Author', 'Cast Member', 'Commenter', 'Composer', 'Cosponsor', 'Counsel', 'Interviewer', 'Producer', 'Recipient', 'Reviewed Author', 'Scriptwriter', 'Words By', 'Guest', 'Number', 'Edition', 'Running Time', 'Scale', 'Medium', 'Artwork Size', 'Filing Date', 'Application Number', 'Assignee', 'Issuing Authority', 'Country', 'Meeting

,PMID,Item Type,Publication Year,Author,Title,Publication Title,ISBN,ISSN,DOI,Url,...,Session,Committee,History,Legislative Body,Embedding,Cluster,TSNE_1,TSNE_2,Cluster_Size,Cluster_Density
0,33YAY7PD,journalArticle,1981.0,"Agobe, J. T.; Good, W.; Hancock, K. W.",Meteorological Relations of Eclampsia in Lagos...,BJOG: An International Journal of Obstetrics &...,NaN,NaN,10.1111/j.1471-0528.1981.tb01269.x,NaN,...,NaN,NaN,NaN,NaN,[ 0.01665183 -0.01438398 0.02183908 ... 0.00...,2,16.016111,10.461332,95,0.299685
1,XVKSWMQY,journalArticle,2010.0,"Algert, Charles S.; Roberts, Christine L.; Sha...",Seasonal variation in pregnancy hypertension i...,American Journal of Obstetrics and Gynecology,NaN,NaN,10.1016/j.ajog.2010.04.020,http://dx.doi.org/10.1016/j.ajog.2010.04.020,...,NaN,NaN,NaN,NaN,[-0.02157679 0.01930683 0.07293151 ... -0.01...,2,9.436985,5.769521,95,0.299685
2,XK47NMBF,journalArticle,2017.0,"Auger, Nathalie; Siemiatycki, Jack; Bilodeau-B...",Ambient Temperature and Risk of Preeclampsia: ...,Paediatric and Perinatal Epidemiology,NaN,NaN,10.1111/ppe.12362,NaN,...,NaN,NaN,NaN,NaN,[ 0.03013899 0.01068883 0.07490356 ... -0.03...,2,9.594197,0.849804,95,0.299685
3,LBG98YBX,journalArticle,2021.0,"Bogan, Mustafa; Al, Behcet; Kul, Seval; Zengin...","The effects of desert dust storms, air polluti...",International Journal of Biometeorology,NaN,NaN,10.1007/s00484-021-02127-8,NaN,...,NaN,NaN,NaN,NaN,[ 0.00519587 0.00982096 0.04526034 ... 0.01...,0,1.121411,6.885093,76,0.239748
4,8D76BC26,journalArticle,2015.0,"Cho, G. J.; Ahn, K. H.; Kim, L. Y.; Hwang, S. ...",Effect of relative humidity on preeclampsia,Clinical and Experimental Obstetrics and Gynec...,NaN,NaN,10.12891/ceog3462.2017,NaN,...,NaN,NaN,NaN,NaN,[-0.01043425 0.01173924 0.05356741 ... -0.01...,2,15.809174,4.780218,95,0.299685


## 5. AI Screening Function with Structured Outputs

Using OpenAI's latest API with native Pydantic support for guaranteed structured responses.

In [63]:
def screen_abstract_with_ai(
    title: str,
    abstract: str,
    criteria: str,
    model: str = "gpt-5",  # GPT-5 reasoning model
    reasoning_effort: str = "medium",  # minimal, low, medium, high
    max_retries: int = 3
) -> dict:
    """
    Screen a single abstract using OpenAI API with structured outputs.
    
    Uses the latest OpenAI SDK (>=1.0.0) with Pydantic models for guaranteed
    structured responses. GPT-5 uses reasoning_effort instead of temperature.
    
    Args:
        title: Article title
        abstract: Article abstract
        criteria: Inclusion/exclusion criteria text
        model: OpenAI model to use (gpt-5, gpt-5-mini, gpt-5-nano, etc.)
        reasoning_effort: Reasoning depth (minimal, low, medium, high)
        max_retries: Maximum number of retry attempts
    
    Returns:
        Dictionary with screening decision and metadata
    """
    
    # Construct the prompt
    # GPT-5 supports "developer" messages (functionally same as "system")
    developer_prompt = f"""You are an expert systematic reviewer conducting abstract screening for a systematic review.

Your task is to carefully evaluate each abstract against the inclusion/exclusion criteria and make a decision to INCLUDE, EXCLUDE, or mark as UNCERTAIN.

INCLUSION/EXCLUSION CRITERIA:
{criteria}

INSTRUCTIONS:
1. Read the title and abstract carefully
2. Evaluate against each criterion (population, exposure, outcome, study design)
3. Make a clear decision: include, exclude, or uncertain
4. If excluding, provide the primary PRISMA-aligned reason
5. Provide brief reasoning (1-3 sentences)
6. Be conservative - if uncertain whether to include, mark as 'uncertain' rather than exclude

Your response will be parsed into a structured format following the ScreeningDecision schema."""
    
    user_prompt = f"""Title: {title}

Abstract: {abstract}

Please screen this abstract and provide your structured decision."""
    
    for attempt in range(max_retries):
        try:
            # Use the beta.chat.completions.parse method for structured outputs
            # GPT-5 models support structured outputs with Pydantic
            completion = client.beta.chat.completions.parse(
                model=model,
                messages=[
                    {"role": "developer", "content": developer_prompt},  # GPT-5 uses "developer" role
                    {"role": "user", "content": user_prompt}
                ],
                response_format=ScreeningDecision,  # Pydantic model for structured output
                reasoning_effort=reasoning_effort,  # GPT-5 uses reasoning_effort instead of temperature
                # Note: temperature is deprecated for GPT-5 models
            )
            
            # Extract the parsed Pydantic object
            screening_result = completion.choices[0].message.parsed
            
            # Convert to dictionary for downstream processing
            return screening_result.model_dump()
                
        except Exception as e:
            wait_time = 2 ** attempt
            logging.warning(f"Error on attempt {attempt + 1}/{max_retries}: {e}")
            
            if attempt < max_retries - 1:
                logging.info(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                logging.error(f"Failed after {max_retries} attempts")
    
    # Fallback if all retries failed
    return {
        "decision": "uncertain",
        "confidence": "low",
        "exclusion_reason": None,
        "reasoning": "Failed to get AI response after multiple attempts",
        "population_match": False,
        "exposure_match": False,
        "outcome_match": False,
        "study_design_appropriate": False
    }

logging.info("AI screening function defined with structured outputs for GPT-5")

2026-01-23 14:43:48,380 - INFO - AI screening function defined with structured outputs for GPT-5


## 6. Model Selection Guide

Choose the right GPT-5 model and reasoning effort for your use case:

### GPT-5 Model Options

| Model | Speed | Cost | Quality | Best For |
|-------|-------|------|---------|----------|
| **gpt-5** | ⚡⚡ Medium | 💰💰 Moderate | ⭐⭐⭐⭐⭐ Excellent | High-accuracy screening, complex criteria |
| **gpt-5-mini** | ⚡⚡⚡ Fast | 💰 Cheap | ⭐⭐⭐⭐ Very Good | Large-scale screening, good balance |
| **gpt-5-nano** | ⚡⚡⚡⚡ Very Fast | 💰 Very Cheap | ⭐⭐⭐ Good | Bulk screening, tight budgets |

### Reasoning Effort Levels

| Reasoning Effort | Speed | Cost | Depth | Best For |
|------------------|-------|------|-------|----------|
| **minimal** | ⚡⚡⚡⚡ Fastest | 💰 Lowest | Shallow | Simple transforms, bulk operations |
| **low** | ⚡⚡⚡ Fast | 💰 Low | Light | Simple screening, quick judgment |
| **medium** (default) | ⚡⚡ Moderate | 💰💰 Medium | Moderate | Most screening tasks |
| **high** | ⚡ Slowest | 💰💰💰 Highest | Deep | Complex cases, difficult decisions |

**Recommendation:** 
- Start with `gpt-5-mini` with `medium` reasoning effort for testing
- Use `gpt-5` with `medium` or `high` reasoning effort for production screening
- Use `low` or `minimal` effort for simple, clear-cut decisions to save costs

## 7. Test Single Abstract

In [7]:
# Test on a single abstract first
test_idx = 0
test_row = df_pubmed.iloc[test_idx]

print(f"Testing on PMID: {test_row['PMID']}")
print(f"\nTitle: {test_row['Title']}")
print(f"\nAbstract (first 300 chars): {test_row['Abstract'][:300]}...")

# Screen the abstract using GPT-5 model
print("\n" + "="*80)
print("Running AI screening with GPT-5 (medium reasoning effort)...")
print("="*80)

test_result = screen_abstract_with_ai(
    title=test_row['Title'],
    abstract=test_row['Abstract'],
    criteria=INCLUSION_CRITERIA,
    model="gpt-5",  # Use GPT-5 reasoning model
    reasoning_effort="medium"  # Balanced reasoning depth
)

print("\n" + "="*80)
print("SCREENING RESULT:")
print("="*80)
print(json.dumps(test_result, indent=2))

# Show the structured output
print("\n" + "="*80)
print("DECISION SUMMARY:")
print("="*80)
print(f"Decision: {test_result['decision'].upper()}")
print(f"Confidence: {test_result['confidence'].upper()}")
if test_result['exclusion_reason']:
    print(f"Exclusion Reason: {test_result['exclusion_reason']}")
print(f"\nReasoning: {test_result['reasoning']}")
print(f"\nCriteria Matches:")
print(f"  ✓ Population: {'Yes' if test_result['population_match'] else 'No'}")
print(f"  ✓ Exposure: {'Yes' if test_result['exposure_match'] else 'No'}")
print(f"  ✓ Outcome: {'Yes' if test_result['outcome_match'] else 'No'}")
print(f"  ✓ Study Design: {'Yes' if test_result['study_design_appropriate'] else 'No'}")

Testing on PMID: 33YAY7PD

Title: Meteorological Relations of Eclampsia in Lagos, Nigeria

Abstract (first 300 chars): A retrospective study of the meteorological relations of eclampsia in Lagos, Nigeria supports other observations that the incidence of this disease varies significantly with the weather. Protective action by arid conditions is consistent with the known effect of dehydration on convulsions of differi...

Running AI screening with GPT-5 (medium reasoning effort)...


2026-01-23 10:00:40,533 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



SCREENING RESULT:
{
  "decision": "uncertain",
  "confidence": "low",
  "exclusion_reason": null,
  "reasoning": "Retrospective study of eclampsia in pregnant women examines meteorological conditions, mentioning cool vs arid conditions, which likely involves ambient temperature. However, the abstract does not clearly specify temperature/heat exposure measures or comparisons across temperature levels. Full text needed to confirm exposure definition and analysis.",
  "population_match": true,
  "exposure_match": true,
  "outcome_match": true,
  "study_design_appropriate": true
}

DECISION SUMMARY:
Decision: UNCERTAIN
Confidence: LOW

Reasoning: Retrospective study of eclampsia in pregnant women examines meteorological conditions, mentioning cool vs arid conditions, which likely involves ambient temperature. However, the abstract does not clearly specify temperature/heat exposure measures or comparisons across temperature levels. Full text needed to confirm exposure definition and analys

## 8. Batch Screening with Progress Tracking

In [64]:
def batch_screen_abstracts(
    df: pd.DataFrame,
    criteria: str,
    output_file: str = "outputs/ai_screening_results.csv",
    batch_size: int = 50,
    start_idx: int = 0,
    rate_limit_delay: float = 1.0,
    model: str = "gpt-5",
    reasoning_effort: str = "medium"
) -> pd.DataFrame:
    """
    Screen all abstracts in batches with progress saving.
    
    Args:
        df: DataFrame with PubMed records
        criteria: Inclusion/exclusion criteria
        output_file: Path to save results
        batch_size: Save progress every N records
        start_idx: Index to start from (for resuming)
        rate_limit_delay: Seconds to wait between API calls
        model: GPT-5 model to use (gpt-5, gpt-5-mini, gpt-5-nano)
        reasoning_effort: Reasoning depth (minimal, low, medium, high)
    
    Returns:
        DataFrame with screening results
    """
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    # Check if output file exists to resume
    if os.path.exists(output_file) and start_idx == 0:
        df_existing = pd.read_csv(output_file)
        processed_pmids = set(df_existing['PMID'].astype(str))
        logging.info(f"Found {len(processed_pmids)} already processed records")
    else:
        processed_pmids = set()
        df_existing = None
    
    # Filter to unprocessed records
    df['PMID'] = df['PMID'].astype(str)
    df_to_process = df[~df['PMID'].isin(processed_pmids)].copy()
    
    logging.info(f"Processing {len(df_to_process)} records with {model} (reasoning_effort={reasoning_effort})")
    
    results = []
    
    # Process with progress bar
    for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Screening abstracts"):
        
        # Screen the abstract
        screening_result = screen_abstract_with_ai(
            title=row['Title'],
            abstract=row['Abstract'],
            criteria=criteria,
            model=model,
            reasoning_effort=reasoning_effort
        )
        
        # Combine with original data
        result_row = {
            'PMID': row['PMID'],
            'Title': row['Title'],
            'Abstract': row['Abstract'],
            'Authors': row.get('Authors', ''),
            'Date': row.get('Date', ''),
            'Journal': row.get('Journal', ''),
            'ai_decision': screening_result['decision'],
            'ai_confidence': screening_result['confidence'],
            'ai_exclusion_reason': screening_result.get('exclusion_reason'),
            'ai_reasoning': screening_result['reasoning'],
            'ai_population_match': screening_result['population_match'],
            'ai_exposure_match': screening_result['exposure_match'],
            'ai_outcome_match': screening_result['outcome_match'],
            'ai_study_design_appropriate': screening_result['study_design_appropriate'],
            'screened_datetime': datetime.now().isoformat(),
            'ai_model': model,
            'ai_reasoning_effort': reasoning_effort
        }
        
        results.append(result_row)
        
        # Save progress every batch_size records
        if len(results) % batch_size == 0:
            df_results = pd.DataFrame(results)
            
            # Append to existing or create new
            if df_existing is not None:
                df_combined = pd.concat([df_existing, df_results], ignore_index=True)
            else:
                df_combined = df_results
            
            df_combined.to_csv(output_file, index=False)
            logging.info(f"Saved progress: {len(df_combined)} total records")
            
            # Update for next batch
            df_existing = df_combined
            results = []
        
        # Rate limiting
        time.sleep(rate_limit_delay)
    
    # Save any remaining results
    if results:
        df_results = pd.DataFrame(results)
        
        if df_existing is not None:
            df_combined = pd.concat([df_existing, df_results], ignore_index=True)
        else:
            df_combined = df_results
        
        df_combined.to_csv(output_file, index=False)
        logging.info(f"Final save: {len(df_combined)} total records")
    else:
        df_combined = df_existing if df_existing is not None else pd.DataFrame()
    
    return df_combined

logging.info("Batch screening function defined")

2026-01-23 14:43:59,441 - INFO - Batch screening function defined


## 9. Run Screening on Subset (for testing)

In [17]:
# Test on first 10 records
df_test = df_pubmed.head(10).copy()

print(f"Testing on {len(df_test)} records with GPT-5...")
print(f"Model: gpt-5 | Reasoning effort: medium")
print(f"Estimated cost: ~$0.05-0.10 (varies with reasoning tokens)")
print("="*80)

df_test_results = batch_screen_abstracts(
    df=df_test,
    criteria=INCLUSION_CRITERIA,
    output_file="outputs/ai_screening_test_results.csv",
    batch_size=5,
    rate_limit_delay=0.5,
    model="gpt-5",  # GPT-5 reasoning model
    reasoning_effort="medium"  # Balanced reasoning depth
)

print("\n" + "="*80)
print("✓ Test screening complete!")
print("="*80)
print(f"Results saved to: outputs/ai_screening_test_results.csv")
print(f"\nQuick Summary:")
print(f"  - Total screened: {len(df_test_results)}")
print(f"  - Included: {(df_test_results['ai_decision'] == 'include').sum()}")
print(f"  - Excluded: {(df_test_results['ai_decision'] == 'exclude').sum()}")
print(f"  - Uncertain: {(df_test_results['ai_decision'] == 'uncertain').sum()}")

df_test_results.head()

2025-11-06 23:54:34,698 - INFO - Processing 10 records with gpt-5 (reasoning_effort=medium)


Testing on 10 records with GPT-5...
Model: gpt-5 | Reasoning effort: medium
Estimated cost: ~$0.05-0.10 (varies with reasoning tokens)


Screening abstracts:  40%|████      | 4/10 [00:32<00:47,  7.92s/it]2025-11-06 23:55:14,303 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-06 23:55:14,320 - INFO - Saved progress: 5 total records
Screening abstracts:  90%|█████████ | 9/10 [01:14<00:08,  8.75s/it]2025-11-06 23:55:56,638 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-06 23:55:56,662 - INFO - Saved progress: 10 total records
Screening abstracts: 100%|██████████| 10/10 [01:22<00:00,  8.25s/it]


✓ Test screening complete!
Results saved to: outputs/ai_screening_test_results.csv

Quick Summary:
  - Total screened: 10
  - Included: 1
  - Excluded: 9
  - Uncertain: 0


,PMID,Title,Abstract,Authors,Date,Journal,ai_decision,ai_confidence,ai_exclusion_reason,ai_reasoning,ai_population_match,ai_exposure_match,ai_outcome_match,ai_study_design_appropriate,screened_datetime,ai_model,ai_reasoning_effort
0,37495151,Influence of temperature on the risk of gestat...,Previous studies have proved that exposure to ...,"Qian Nianfeng, Xu Rongrong, Wei Yongjie, Li Zh...",2023.0,The Science of the total environment,include,high,None,Pregnant population with ambient temperature e...,True,True,True,True,2025-11-06T23:54:42.878881,gpt-5,medium
1,33859477,"Physical Activity in Pregnancy: Beliefs, Benef...",Notwithstanding the benefits of physical activ...,"Okafor Uchenna Benedine, Goon Daniel Ter",2021.0,Journal of multidisciplinary healthcare,exclude,high,wrong_exposure,The study focuses on beliefs and information-s...,True,False,False,True,2025-11-06T23:54:50.110548,gpt-5,medium
2,40898443,Joint Contribution of Mid-Trimester Systolic a...,To examine how mid-trimester systolic blood pr...,"Fang Yiwen, Chen Huaxi, Yang Jingbo, Zhang Rui...",2025.0,American journal of hypertension,exclude,high,wrong_exposure,Cohort study in pregnant women examines mid-tr...,True,False,False,True,2025-11-06T23:55:00.698385,gpt-5,medium
3,26187609,Serum HtrA1 is differentially regulated betwee...,HtrA1 (high temperature requirement A1) is a s...,"Teoh Sonia Soo Yee, Zhao Min, Wang Yao, Chen Q...",2015.0,Placenta,exclude,high,wrong_exposure,The study examines a biomarker (HtrA1) in rela...,True,False,True,True,2025-11-06T23:55:06.837540,gpt-5,medium
4,24860691,Differential Expression of HrtA1 and ADAM12 in...,High temperature requirement factor A 1 (HtrA1...,"Enquobahrie Daniel A, Hevner Karin, Qiu Chunfa...",2012.0,Reproductive system & sexual disorders : curre...,exclude,high,wrong_exposure,This study examines placental expression of Ht...,True,False,True,True,2025-11-06T23:55:14.317616,gpt-5,medium


## 10. Run Full Screening

In [65]:
#
# # WARNING: This will make many API calls and may take hours/days depending on dataset size
# # GPT-5 costs vary based on reasoning tokens used

print(f"Starting full screening of {len(df_pubmed)} records...")
print("This may take a long time and cost money. Press Ctrl+C to cancel.")
df_full_results = batch_screen_abstracts(
    df=df_pubmed,
    criteria=INCLUSION_CRITERIA,
    output_file="outputs/ai_screening_full_results.csv",
    batch_size=50,
    rate_limit_delay=1.0,  # Adjust based on your rate limits
    model="gpt-5",  # Use gpt-5, gpt-5-mini, or gpt-5-nano
    reasoning_effort="medium"  # minimal, low, medium, or high
)
print("\nFull screening complete!")
print(f"Results saved to: outputs/ai_screening_full_results.csv")

2026-01-23 14:44:14,508 - INFO - Processing 237 records with gpt-5 (reasoning_effort=medium)


Starting full screening of 237 records...
This may take a long time and cost money. Press Ctrl+C to cancel.


Screening abstracts:  21%|██        | 49/237 [09:41<39:04, 12.47s/it]2026-01-23 14:54:17,580 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-23 14:54:17,591 - INFO - Saved progress: 50 total records
Screening abstracts:  42%|████▏     | 99/237 [26:50<26:30, 11.53s/it]2026-01-23 15:11:12,467 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-23 15:11:12,477 - INFO - Saved progress: 100 total records
Screening abstracts:  63%|██████▎   | 149/237 [35:21<20:13, 13.79s/it]2026-01-23 15:19:44,231 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-23 15:19:44,253 - INFO - Saved progress: 150 total records
Screening abstracts:  84%|████████▍ | 199/237 [46:32<09:17, 14.66s/it]2026-01-23 15:30:54,865 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-23 15:30:54,886 - INFO - Saved progress: 200 total records
Scr


Full screening complete!
Results saved to: outputs/ai_screening_full_results.csv


In [66]:
df_temp_1 = pd.read_csv('outputs/ai_screening_full_results.csv')

In [67]:
df_temp_2 = pd.read_csv('/Users/nicholas/Documents/Work/WPHR/Development/ml_review_heat_PIH/scripts/outputs/labeled_clusters_subset_2.csv')
df_temp_2.shape

(237, 93)

In [69]:
#len(df_temp_1)
len(df_temp_2)
df_temp_2.shape

(237, 93)

## 11. Load Embeddings and Merge with Screening Results

**Understanding the Two-Stage Screening:**

1. **Complete Dataset** (`pubmed_results_with_embeddings.csv`):
   - Contains ALL PubMed records (e.g., 10,000 records)
   - Each has an embedding vector for ML analysis

2. **ML (KNN) Pre-Filtering**:
   - Some records were filtered out by KNN similarity
   - These records are NOT in `ai_screening_full_results.csv`
   - They never went through LLM screening

3. **LLM (GPT-5) Screening** (`ai_screening_full_results.csv`):
   - Contains only records that passed ML filter
   - Each has an `ai_decision`: include/exclude/uncertain
   - Each has `ai_confidence`, `ai_exclusion_reason`, etc.

4. **Merging Strategy**:
   - Left join: Complete embeddings ← LLM screening results
   - Records WITH screening data → LLM decisions
   - Records WITHOUT screening data → Labeled as 'screened_by_ml'

This allows us to visualize the complete dataset showing both ML-filtered and LLM-screened records.

In [70]:
import os
def load_and_merge_with_embeddings(
    screening_file: str = "outputs/ai_screening_full_results.csv",
    embeddings_file: str = "outputs/csv_results_with_embeddings.csv"
):
    """
    Load screening results and merge with original embeddings for visualization.
    """
    # Load screening results
    df_screening = pd.read_csv(screening_file)
    logging.info(f"Loaded {len(df_screening)} screening results")
    
    # Load embeddings
    df_embeddings = pd.read_csv(embeddings_file)
    logging.info(f"Loaded {len(df_embeddings)} records with embeddings")
    
    # Ensure PMID is string for merging
    df_screening['PMID'] = df_screening['PMID'].astype(str)
    df_embeddings['PMID'] = df_embeddings['PMID'].astype(str)
    
    # Merge on PMID
    df_merged = df_embeddings.merge(
        df_screening[['PMID', 'ai_decision', 'ai_confidence', 'ai_exclusion_reason',
                      'ai_population_match', 'ai_exposure_match', 'ai_outcome_match',
                      'ai_study_design_appropriate']],
        on='PMID',
        how='left'
    )
    
    # CRITICAL: Fill NaN for records NOT in screening file (these were excluded by ML)
    # If ai_decision is NaN, it means this PMID was NOT in ai_screening_full_results.csv
    # Therefore, it was pre-filtered by ML (KNN) and never sent to LLM
    df_merged['ai_decision'] = df_merged['ai_decision'].fillna('screened_by_ml')
    df_merged['ai_confidence'] = df_merged['ai_confidence'].fillna('ml_filtered')
    df_merged['screening_method'] = df_merged.apply(
        lambda row: 'LLM (GPT-5)' if row['ai_decision'] not in ['screened_by_ml'] else 'ML (KNN)',
        axis=1
    )
    
    # Verification logging
    logging.info(f"Merged dataset has {len(df_merged)} records")
    logging.info(f"  → Screened by LLM: {(df_merged['ai_decision'].isin(['include', 'exclude', 'uncertain'])).sum()}")
    logging.info(f"  → Screened by ML (KNN): {(df_merged['ai_decision'] == 'screened_by_ml').sum()}")
    
    # Double-check the math
    n_llm = (df_merged['ai_decision'].isin(['include', 'exclude', 'uncertain'])).sum()
    n_ml = (df_merged['ai_decision'] == 'screened_by_ml').sum()
    assert n_llm + n_ml == len(df_merged), "ERROR: Numbers don't add up!"
    assert n_llm == len(df_screening), "ERROR: LLM count doesn't match screening file!"
    
    return df_merged

# Load and merge data - VERIFY THE LOGIC

if os.path.exists("outputs/ai_screening_full_results.csv"):
    print("=" * 80)
    print("LOADING AND MERGING DATA")
    print("=" * 80)
    
    df_merged = load_and_merge_with_embeddings(
        "outputs/ai_screening_full_results.csv",
        "outputs/csv_results_with_embeddings.csv"
    )
    
    print(f"\n✓ Merged dataset: {len(df_merged):,} records")
    print(f"\n📊 Decision distribution:")
    decision_dist = df_merged['ai_decision'].value_counts()
    for decision, count in decision_dist.items():
        pct = count / len(df_merged) * 100
        label = decision.replace('_', ' ').title()
        print(f"   • {label:20s}: {count:6,} ({pct:5.1f}%)")
    
    print("\n✓ Data merge verified - ready for visualization")
else:
    print("No results file found. Run screening first.")

2026-01-23 15:43:30,180 - INFO - Loaded 237 screening results


LOADING AND MERGING DATA


2026-01-23 15:43:30,613 - INFO - Loaded 2920 records with embeddings
2026-01-23 15:43:30,632 - INFO - Merged dataset has 2920 records
2026-01-23 15:43:30,634 - INFO -   → Screened by LLM: 237
2026-01-23 15:43:30,635 - INFO -   → Screened by ML (KNN): 2683



✓ Merged dataset: 2,920 records

📊 Decision distribution:
   • Screened By Ml      :  2,683 ( 91.9%)
   • Exclude             :    185 (  6.3%)
   • Include             :     46 (  1.6%)
   • Uncertain           :      6 (  0.2%)

✓ Data merge verified - ready for visualization


## 12. Analyze Screening Results with Statistics

In [71]:
def analyze_screening_results(results_file: str = "outputs/ai_screening_full_results.csv"):
    """
    Generate comprehensive summary statistics and performance metrics.
    """
    df = pd.read_csv(results_file)
    
    print("="*80)
    print("SCREENING RESULTS SUMMARY")
    print("="*80)
    print(f"\nTotal records analyzed: {len(df)}")
    print(f"Screened by LLM (GPT-5): {len(df)}")
    
    # Decision breakdown
    print("\n" + "="*80)
    print("DECISION BREAKDOWN (LLM SCREENING)")
    print("="*80)
    decision_counts = df['ai_decision'].value_counts()
    for decision, count in decision_counts.items():
        pct = count / len(df) * 100
        print(f"{decision.upper():12s}: {count:5d} ({pct:5.1f}%)")
    
    inclusion_rate = (df['ai_decision'] == 'include').sum() / len(df) * 100
    print(f"\nInclusion rate: {inclusion_rate:.1f}%")
    print(f"\nNote: Records not shown here were pre-filtered by ML (KNN)")
    
    # Confidence levels
    print("\n" + "="*80)
    print("CONFIDENCE LEVELS")
    print("="*80)
    conf_counts = df['ai_confidence'].value_counts()
    for conf, count in conf_counts.items():
        pct = count / len(df) * 100
        print(f"{conf.upper():12s}: {count:5d} ({pct:5.1f}%)")
    
    # Exclusion reasons
    print("\n" + "="*80)
    print("EXCLUSION REASONS")
    print("="*80)
    df_excluded = df[df['ai_decision'] == 'exclude']
    if len(df_excluded) > 0:
        excl_counts = df_excluded['ai_exclusion_reason'].value_counts()
        for reason, count in excl_counts.items():
            pct = count / len(df_excluded) * 100
            print(f"{str(reason):30s}: {count:4d} ({pct:5.1f}%)")
    else:
        print("No exclusions")
    
    # Criteria matching
    print("\n" + "="*80)
    print("CRITERIA MATCHING RATES")
    print("="*80)
    criteria = ['ai_population_match', 'ai_exposure_match', 'ai_outcome_match', 'ai_study_design_appropriate']
    labels = ['Population', 'Exposure', 'Outcome', 'Study Design']
    
    for label, col in zip(labels, criteria):
        count = df[col].sum()
        pct = count / len(df) * 100
        print(f"{label:20s}: {count:5d} ({pct:5.1f}%)")
    
    # Confidence by decision
    print("\n" + "="*80)
    print("CONFIDENCE BY DECISION")
    print("="*80)
    conf_by_decision = pd.crosstab(df['ai_decision'], df['ai_confidence'], normalize='index') * 100
    print(conf_by_decision.round(1))
    
    # PRISMA flow numbers
    print("\n" + "="*80)
    print("PRISMA FLOW DIAGRAM NUMBERS")
    print("="*80)
    print(f"Records identified:                    {len(df):5d}")
    print(f"Records screened:                      {len(df):5d}")
    print(f"Records excluded:                      {(df['ai_decision'] == 'exclude').sum():5d}")
    print(f"Records included:                      {(df['ai_decision'] == 'include').sum():5d}")
    print(f"Records uncertain (manual review):     {(df['ai_decision'] == 'uncertain').sum():5d}")
    
    return df

# Analyze results
# Uncomment to run:
if os.path.exists("outputs/ai_screening_full_results.csv"):
     df_analysis = analyze_screening_results("outputs/ai_screening_full_results.csv")
else:
   print("No results file found. Run screening first.")

SCREENING RESULTS SUMMARY

Total records analyzed: 237
Screened by LLM (GPT-5): 237

DECISION BREAKDOWN (LLM SCREENING)
EXCLUDE     :   185 ( 78.1%)
INCLUDE     :    46 ( 19.4%)
UNCERTAIN   :     6 (  2.5%)

Inclusion rate: 19.4%

Note: Records not shown here were pre-filtered by ML (KNN)

CONFIDENCE LEVELS
HIGH        :   212 ( 89.5%)
MEDIUM      :    19 (  8.0%)
LOW         :     6 (  2.5%)

EXCLUSION REASONS
wrong_exposure                :   59 ( 31.9%)
wrong_study_design            :   58 ( 31.4%)
wrong_outcome                 :   53 ( 28.6%)
wrong_population              :    9 (  4.9%)
animal_study                  :    3 (  1.6%)
no_original_data              :    3 (  1.6%)

CRITERIA MATCHING RATES
Population          :   197 ( 83.1%)
Exposure            :   120 ( 50.6%)
Outcome             :   122 ( 51.5%)
Study Design        :   156 ( 65.8%)

CONFIDENCE BY DECISION
ai_confidence  high    low  medium
ai_decision                       
exclude        98.4    0.0     1.6
include

## 13. Create Visualization Functions

**Enhanced Visualization Suite (v2.0):**

This section creates comprehensive visualizations with the following improvements:

1. **Complete Screening Pipeline Visibility**: 
   - **ML-Excluded shown separately**: Grey nodes/flows for KNN pre-filtered records
   - **LLM decisions branched properly**: Excluded/Uncertain/Included shown as distinct paths
   - **Sankey diagram**: Proper flow showing all decision points with percentages

2. **Interactive t-SNE with ALL Categories**:
   - **4 toggleable categories**: 
     * Excluded by ML (KNN) - Grey, smallest markers
     * Excluded by LLM - Red markers with reasons
     * Uncertain (LLM) - Orange markers
     * Included (LLM) - Green, largest markers
   - **Click legend** to show/hide any category
   - **Different marker sizes** for visual hierarchy

3. **Detailed Exclusion Reasons**: 
   - Separate visualization showing why LLM excluded studies
   - Counts, percentages, and detailed breakdown
   - Hover for full statistics (% of excluded, % of total)

4. **Static Highlighted View**: 
   - Included studies prominently shown in green
   - All others greyed out for clear focus on final included set

5. **Method-Aware Throughout**: 
   - All visualizations distinguish ML (KNN) from LLM (GPT-5)
   - Hover tooltips show screening method and confidence
   - Color-coded by screening stage and decision

## 13. Visualization Functions

Create comprehensive visualizations using Plotly and Matplotlib.

In [72]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import os
import logging
import pandas as pd
import numpy as np

# Set style for matplotlib
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

logging.info("Visualization libraries loaded")

2026-01-23 15:48:09,404 - INFO - Visualization libraries loaded


In [73]:
def create_funnel_plot(df, output_dir="outputs/visualizations"):
    """
    Create PRISMA-style funnel plot showing screening flow.
    Note: df contains only LLM-screened records.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Calculate numbers for each stage (LLM screening only)
    n_llm_screened = len(df)
    n_excluded = (df['ai_decision'] == 'exclude').sum()
    n_uncertain = (df['ai_decision'] == 'uncertain').sum()
    n_included = (df['ai_decision'] == 'include').sum()
    
    # Create funnel plot using Plotly
    fig = go.Figure()
    
    # Funnel stages for LLM screening
    stages = ['LLM Screened<br>(GPT-5)', 'After Exclusions', 'Final Included']
    values = [n_llm_screened, n_included + n_uncertain, n_included]
    
    fig.add_trace(go.Funnel(
        y=stages,
        x=values,
        textposition="inside",
        textinfo="value+percent initial",
        marker=dict(
            color=['#3498db', '#f39c12', '#27ae60'],
            line=dict(width=2, color='white')
        ),
        connector=dict(line=dict(color='gray', width=3))
    ))
    
    fig.update_layout(
        title={
            'text': 'LLM Screening Flow',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'family': 'Arial Black'}
        },
        height=650,
        showlegend=False,
        annotations=[
            dict(
                text=f"<b>LLM Screening Results:</b><br>" +
                     f"Excluded: {n_excluded} ({n_excluded/n_llm_screened*100:.1f}%)<br>" +
                     f"Uncertain: {n_uncertain} ({n_uncertain/n_llm_screened*100:.1f}%)<br>" +
                     f"Included: {n_included} ({n_included/n_llm_screened*100:.1f}%)",
                xref="paper", yref="paper",
                x=0.95, y=0.5,
                showarrow=False,
                font=dict(size=12),
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#3498db',
                borderwidth=2,
                align='left'
            )
        ]
    )
    
    # Save
    fig.write_html(f"{output_dir}/funnel_plot.html")
    fig.write_image(f"{output_dir}/funnel_plot.pdf")
    logging.info(f"Funnel plot saved to {output_dir}/")
    
    return fig

In [74]:
def create_screening_pipeline_diagram(df_merged, output_dir="outputs/visualizations"):
    """
    Create comprehensive screening pipeline showing ML (KNN) and LLM (GPT-5) stages.
    Shows ML-excluded separately, then LLM decisions with detailed exclusion reasons.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # STEP-BY-STEP CALCULATION
    # 1. Total records in complete dataset
    n_total = len(df_merged)
    print(f"\n   DEBUG: Total records in merged dataset: {n_total:,}")
    
    # 2. ML-excluded: records NOT in ai_screening_full_results.csv (ai_decision is NaN before fillna)
    n_ml_excluded = (df_merged['ai_decision'] == 'screened_by_ml').sum()
    print(f"   DEBUG: Records excluded by ML (KNN): {n_ml_excluded:,}")
    
    # 3. LLM-screened: records that ARE in ai_screening_full_results.csv
    n_llm_screened = n_total - n_ml_excluded
    print(f"   DEBUG: Records screened by LLM (GPT-5): {n_llm_screened:,}")
    
    # 4. LLM decisions breakdown
    n_llm_included = (df_merged['ai_decision'] == 'include').sum()
    n_llm_excluded = (df_merged['ai_decision'] == 'exclude').sum()
    n_llm_uncertain = (df_merged['ai_decision'] == 'uncertain').sum()
    print(f"   DEBUG: LLM decisions - Include: {n_llm_included:,}, Exclude: {n_llm_excluded:,}, Uncertain: {n_llm_uncertain:,}")
    
    # 5. Get ALL exclusion reasons for LLM-excluded (where count >= 1)
    df_llm_excluded = df_merged[df_merged['ai_decision'] == 'exclude']
    exclusion_reasons = df_llm_excluded['ai_exclusion_reason'].value_counts()
    exclusion_reasons = exclusion_reasons[exclusion_reasons >= 1]  # Include all with n>=1
    print(f"   DEBUG: All LLM exclusion reasons ({len(exclusion_reasons)} total): {dict(exclusion_reasons)}")
    
    # Build Sankey diagram with exclusion reason detail - INVERTED layout
    # Node ordering: Included at TOP (y=0), Excluded at BOTTOM (y=1)
    nodes_labels = [
        f"Total<br>n={n_total:,}",                    # 0
        f"Excluded by ML<br>n={n_ml_excluded:,}",    # 1
        f"LLM Screened<br>n={n_llm_screened:,}",     # 2
        f"<b>Included<br>n={n_llm_included:,}</b>",  # 3 - Bold at TOP
        f"Uncertain<br>n={n_llm_uncertain:,}",       # 4
        f"Excluded by LLM<br>n={n_llm_excluded:,}"   # 5 - at BOTTOM
    ]
    
    nodes_colors = ["#3498db", "#b0b0b0", "#2ecc71", "#27ae60", "#f39c12", "#e74c3c"]
    
    # Manual positioning for INVERTED layout (Included at top)
    # x: 0 = left, 1 = right; y: 0 = top, 1 = bottom
    nodes_x = [0.01, 0.1, 0.1, 0.40, 0.40, 0.40]
    nodes_y = [0.5, 0.2, 0.3, 0.03, 0.45, 0.9]  # Included=0.03 (top), Excluded=0.9 (bottom)
    
    # Add nodes for ALL exclusion reasons (dynamically positioned)
    reason_node_start = 6
    num_reasons = len(exclusion_reasons)
    y_spacing = 0.8 / max(num_reasons, 1)  # Distribute evenly in bottom 80% of space
    
    for i, (reason, count) in enumerate(exclusion_reasons.items()):
        reason_short = reason.replace('_', ' ').title()
        if len(reason_short) > 25:
            reason_short = reason_short[:22] + "..."
        nodes_labels.append(f"{reason_short}<br>n={count}")
        nodes_colors.append("#c0392b")  # Darker red for reasons
        nodes_x.append(0.60)
        nodes_y.append(0.15 + i * y_spacing)  # Evenly distribute all reasons
    
    # Links (updated node indices after reordering)
    sources = [0, 0]  # Total → ML Excluded, Total → LLM Screened
    targets = [1, 2]
    values = [n_ml_excluded, n_llm_screened]
    link_colors = ["rgba(176,176,176,0.4)", "rgba(46,204,113,0.4)"]
    
    # From LLM Screened to outcomes (node 3=Included, 4=Uncertain, 5=Excluded)
    if n_llm_included > 0:
        sources.append(2)
        targets.append(3)
        values.append(n_llm_included)
        link_colors.append("rgba(39,174,96,0.5)")
    
    if n_llm_uncertain > 0:
        sources.append(2)
        targets.append(4)
        values.append(n_llm_uncertain)
        link_colors.append("rgba(243,156,18,0.4)")
    
    if n_llm_excluded > 0:
        sources.append(2)
        targets.append(5)
        values.append(n_llm_excluded)
        link_colors.append("rgba(231,76,60,0.4)")
    
    # From LLM Excluded (node 5) to specific reasons
    for i, (reason, count) in enumerate(exclusion_reasons.items()):
        sources.append(5)
        targets.append(reason_node_start + i)
        values.append(count)
        link_colors.append("rgba(192,57,43,0.3)")
    
    # Create Sankey with INVERTED layout and text positioned to the right of bars
    fig = go.Figure(data=[go.Sankey(
        arrangement='snap',
        node=dict(
            pad=8,
            thickness=30,
            line=dict(width=0),  # Remove black box around labels
            label=nodes_labels,
            color=nodes_colors,
            x=nodes_x,
            y=nodes_y,
            align='right'  # Position text to the right of bars
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
        ),
        textfont=dict(size=10, family="Arial", color="black"),
        valueformat=",",
        valuesuffix=" studies"
    )])
    
    fig.update_layout(
        title={
            'text': 'Screening Pipeline',
            'x': 0.1,
            'xanchor': 'center',
            'font': {'size': 18}
        },
        height=550,
        width=1100,
        font=dict(size=10),
        margin=dict(l=10, r=150, t=60, b=10),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    # Save
    fig.write_html(f"{output_dir}/screening_pipeline.html")
    fig.write_image(f"{output_dir}/screening_pipeline.pdf")
    logging.info(f"Screening pipeline diagram saved to {output_dir}/")
    
    return fig

def create_decision_breakdown_plots(df, output_dir="outputs/visualizations"):
    """
    Create comprehensive decision breakdown visualizations.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Decision Distribution', 'Confidence Levels',
                       'Exclusion Reasons', 'Criteria Matching'),
        specs=[[{"type": "pie"}, {"type": "pie"}],
               [{"type": "bar"}, {"type": "bar"}]]
    )
    
    # 1. Decision distribution
    decision_counts = df['ai_decision'].value_counts()
    colors_decision = {'include': '#27ae60', 'exclude': '#e74c3c', 'uncertain': '#f39c12'}
    fig.add_trace(
        go.Pie(
            labels=decision_counts.index,
            values=decision_counts.values,
            marker=dict(colors=[colors_decision.get(x, '#95a5a6') for x in decision_counts.index]),
            textinfo='label+percent',
            textposition='inside'
        ),
        row=1, col=1
    )
    
    # 2. Confidence levels
    conf_counts = df['ai_confidence'].value_counts()
    colors_conf = {'high': '#27ae60', 'medium': '#f39c12', 'low': '#e74c3c'}
    fig.add_trace(
        go.Pie(
            labels=conf_counts.index,
            values=conf_counts.values,
            marker=dict(colors=[colors_conf.get(x, '#95a5a6') for x in conf_counts.index]),
            textinfo='label+percent',
            textposition='inside'
        ),
        row=1, col=2
    )
    
    # 3. Exclusion reasons
    df_excluded = df[df['ai_decision'] == 'exclude']
    if len(df_excluded) > 0:
        excl_counts = df_excluded['ai_exclusion_reason'].value_counts().head(10)
        fig.add_trace(
            go.Bar(
                x=excl_counts.values,
                y=excl_counts.index,
                orientation='h',
                marker=dict(color='#e74c3c'),
                text=excl_counts.values,
                textposition='auto'
            ),
            row=2, col=1
        )
    
    # 4. Criteria matching
    criteria_data = {
        'Population': df['ai_population_match'].sum(),
        'Exposure': df['ai_exposure_match'].sum(),
        'Outcome': df['ai_outcome_match'].sum(),
        'Study Design': df['ai_study_design_appropriate'].sum()
    }
    fig.add_trace(
        go.Bar(
            x=list(criteria_data.keys()),
            y=list(criteria_data.values()),
            marker=dict(color='#3498db'),
            text=[f"{v} ({v/len(df)*100:.1f}%)" for v in criteria_data.values()],
            textposition='auto'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text="LLM Screening Results - Comprehensive Breakdown",
        showlegend=False,
        height=900,
        title_font_size=20
    )
    
    fig.update_yaxes(title_text="Exclusion Reason", row=2, col=1)
    fig.update_xaxes(title_text="Count", row=2, col=1)
    fig.update_yaxes(title_text="Count", row=2, col=2)
    fig.update_xaxes(title_text="Criteria", row=2, col=2)
    
    # Save
    fig.write_html(f"{output_dir}/decision_breakdown.html")
    fig.write_image(f"{output_dir}/decision_breakdown.pdf")
    logging.info(f"Decision breakdown plots saved to {output_dir}/")
    
    return fig

def create_exclusion_reasons_detail(df, output_dir="outputs/visualizations"):
    """
    Create detailed visualization of exclusion reasons with counts and percentages.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    df_excluded = df[df['ai_decision'] == 'exclude']
    
    if len(df_excluded) == 0:
        logging.warning("No excluded records to visualize")
        return None
    
    # Get exclusion reason counts
    excl_counts = df_excluded['ai_exclusion_reason'].value_counts()
    
    # Create horizontal bar chart
    fig = go.Figure()
    
    # Calculate percentages
    percentages = (excl_counts / len(df_excluded) * 100).round(1)
    
    # Create labels with counts and percentages
    labels = [reason.replace('_', ' ').title() for reason in excl_counts.index]
    hover_text = [f"<b>{label}</b><br>Count: {count}<br>Percentage: {pct}% of excluded<br>{count/len(df)*100:.1f}% of all LLM-screened"
                  for label, count, pct in zip(labels, excl_counts.values, percentages)]
    
    fig.add_trace(go.Bar(
        y=labels,
        x=excl_counts.values,
        orientation='h',
        marker=dict(
            color=excl_counts.values,
            colorscale='Reds',
            showscale=True,
            colorbar=dict(title="Count")
        ),
        text=[f"{count} ({pct}%)" for count, pct in zip(excl_counts.values, percentages)],
        textposition='outside',
        hovertext=hover_text,
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title={
            'text': f'LLM Exclusion Reasons Detail<br><sub>Total excluded: {len(df_excluded):,} ({len(df_excluded)/len(df)*100:.1f}% of LLM-screened)</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 18}
        },
        xaxis_title='Number of Studies',
        yaxis_title='Exclusion Reason',
        height=max(600, len(excl_counts) * 40),
        width=1000,
        showlegend=False,
        yaxis=dict(autorange="reversed")
    )
    
    # Save
    fig.write_html(f"{output_dir}/exclusion_reasons_detail.html")
    fig.write_image(f"{output_dir}/exclusion_reasons_detail.pdf")
    logging.info(f"Exclusion reasons detail saved to {output_dir}/")
    
    return fig

In [75]:
def create_confidence_heatmap(df, output_dir="outputs/visualizations"):
    """
    Create heatmap showing confidence levels by decision type.
    Uses log transform to better visualize subtleties in lower-count cells.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Cross-tabulation
    conf_matrix = pd.crosstab(df['ai_decision'], df['ai_confidence'])
    
    # Apply log transform for better visualization (log10(x + 1) to handle zeros)
    z_log = np.log10(conf_matrix.values + 1)
    
    # Create custom hover text showing actual counts
    hover_text = []
    for i, decision in enumerate(conf_matrix.index):
        hover_row = []
        for j, confidence in enumerate(conf_matrix.columns):
            count = conf_matrix.values[i, j]
            total_for_decision = conf_matrix.values[i, :].sum()
            pct = (count / total_for_decision * 100) if total_for_decision > 0 else 0
            hover_row.append(
                f"Decision: {decision}<br>" +
                f"Confidence: {confidence}<br>" +
                f"Count: {count:,}<br>" +
                f"% of {decision}: {pct:.1f}%"
            )
        hover_text.append(hover_row)
    
    # Create heatmap with log-transformed data
    fig = go.Figure(data=go.Heatmap(
        z=z_log,
        x=conf_matrix.columns,
        y=conf_matrix.index,
        colorscale='RdYlGn',
        text=conf_matrix.values,  # Show actual counts in cells
        texttemplate='%{text}',
        textfont={"size": 14},
        hovertext=hover_text,
        hoverinfo='text',
        colorbar=dict(
            title="log₁₀(Count + 1)",
            tickmode="linear",
            tick0=0,
            dtick=0.5
        )
    ))
    
    fig.update_layout(
        title={
            'text': 'Confidence Levels by Decision Type',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 18}
        },
        xaxis_title='Confidence Level',
        yaxis_title='Decision',
        height=500,
        width=750,
        title_font_size=18
    )
    
    # Save
    fig.write_html(f"{output_dir}/confidence_heatmap.html")
    fig.write_image(f"{output_dir}/confidence_heatmap.pdf")
    logging.info(f"Confidence heatmap saved to {output_dir}/")
    
    return fig

In [76]:
def prepare_tsne_data(df_merged, perplexity=50, random_state=42):
    """
    Prepare t-SNE embeddings for visualization.
    """
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler
    
    # Parse embeddings
    def safe_convert(val):
        try:
            if isinstance(val, str):
                embedding = np.array(eval(val))
            else:
                embedding = np.array(val)
            return embedding if embedding.size > 0 else None
        except:
            return None
    
    df_merged['Embedding_parsed'] = df_merged['Embedding'].apply(safe_convert)
    df_valid = df_merged.dropna(subset=['Embedding_parsed']).copy()
    
    logging.info(f"Valid embeddings: {len(df_valid)}")
    
    # Convert to array
    embeddings_array = np.array(df_valid['Embedding_parsed'].values.tolist())
    
    # Scale
    scaler = StandardScaler()
    embeddings_scaled = scaler.fit_transform(embeddings_array)
    
    # t-SNE
    perplexity = min(perplexity, len(df_valid) - 1)
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        learning_rate=200,
        max_iter=1000,
        metric='cosine',
        init='pca',
        random_state=random_state
    )
    
    logging.info("Running t-SNE...")
    embeddings_2d = tsne.fit_transform(embeddings_scaled)
    
    df_valid['TSNE_1'] = embeddings_2d[:, 0]
    df_valid['TSNE_2'] = embeddings_2d[:, 1]
    
    # Verify all decision categories are present
    logging.info(f"t-SNE decision breakdown:")
    for decision in ['screened_by_ml', 'exclude', 'uncertain', 'include']:
        count = (df_valid['ai_decision'] == decision).sum()
        logging.info(f"  → {decision}: {count:,}")
    
    return df_valid

In [77]:
def create_tsne_visualization(df_tsne, output_dir="outputs/visualizations"):
    """
    Create interactive t-SNE visualization with decision filtering.
    Interactive HTML shows all decisions with filtering options.
    Also creates a static version highlighting only included studies.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Color mapping
    color_map = {
        'include': '#27ae60',
        'exclude': '#e74c3c',
        'uncertain': '#f39c12',
        'screened_by_ml': '#95a5a6'
    }
    
    df_tsne['color'] = df_tsne['ai_decision'].map(color_map)
    
    # Create interactive figure with all points and filtering
    fig_interactive = go.Figure()
    
    # Add traces for each decision type with screening method info
    # Order: ML screened first (background), then LLM decisions
    decision_order = [
        ('screened_by_ml', 'Excluded by ML (KNN)', '#b0b0b0', 4),
        ('exclude', 'Excluded by LLM', '#e74c3c', 5),
        ('uncertain', 'Uncertain (LLM)', '#f39c12', 6),
        ('include', 'Included (LLM)', '#27ae60', 7)
    ]
    
    for decision, display_label, color, marker_size in decision_order:
        df_subset = df_tsne[df_tsne['ai_decision'] == decision]
        
        if len(df_subset) > 0:
            # Prepare custom data for hover
            if 'screening_method' in df_subset.columns and 'ai_confidence' in df_subset.columns:
                customdata = df_subset[['screening_method', 'ai_confidence']].values
                hover_extra = 'Screening: %{customdata[0]}<br>Confidence: %{customdata[1]}<br>'
            else:
                customdata = None
                hover_extra = ''
            
            fig_interactive.add_trace(go.Scatter(
                x=df_subset['TSNE_1'],
                y=df_subset['TSNE_2'],
                mode='markers',
                name=f"{display_label} (n={len(df_subset):,})",
                marker=dict(
                    size=marker_size,
                    color=color,
                    opacity=0.7 if decision != 'screened_by_ml' else 0.4,
                    line=dict(width=0.5, color='white')
                ),
                text=df_subset['Title'],
                customdata=customdata,
                hovertemplate='<b>%{text}</b><br>' +
                             'Category: ' + display_label + '<br>' +
                             hover_extra +
                             '<extra></extra>',
                visible=True  # All visible by default in interactive
            ))
    
    # Update layout for interactive
    fig_interactive.update_layout(
        title={
            'text': 't-SNE Visualization of Screening Results',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title='t-SNE Dimension 1',
        yaxis_title='t-SNE Dimension 2',
        height=800,
        width=1200,
        hovermode='closest',
        legend=dict(
            title='Decision',
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1
        )
    )
    
    # Save interactive version
    fig_interactive.write_html(f"{output_dir}/tsne_visualization_interactive.html")
    fig_interactive.write_image(f"{output_dir}/tsne_visualization_interactive.pdf")
    logging.info(f"Interactive t-SNE visualization saved to {output_dir}/")
    
    # Create static figure highlighting INCLUDED studies only
    fig_static = go.Figure()
    
    # Add excluded/uncertain/ML screened as grey background
    df_background = df_tsne[df_tsne['ai_decision'] != 'include']
    if len(df_background) > 0:
        fig_static.add_trace(go.Scatter(
            x=df_background['TSNE_1'],
            y=df_background['TSNE_2'],
            mode='markers',
            name='Other Studies',
            marker=dict(
                size=5,
                color='#d3d3d3',
                opacity=0.3,
                line=dict(width=0)
            ),
            text=df_background['Title'],
            hovertemplate='<b>%{text}</b><br>' +
                         'Status: Not Included<br>' +
                         '<extra></extra>',
            showlegend=True
        ))
    
    # Highlight INCLUDED studies
    df_included = df_tsne[df_tsne['ai_decision'] == 'include']
    if len(df_included) > 0:
        fig_static.add_trace(go.Scatter(
            x=df_included['TSNE_1'],
            y=df_included['TSNE_2'],
            mode='markers',
            name=f'Included Studies (n={len(df_included)})',
            marker=dict(
                size=8,
                color='#27ae60',
                opacity=0.9,
                line=dict(width=1, color='white')
            ),
            text=df_included['Title'],
            customdata=df_included[['ai_confidence']].values,
            hovertemplate='<b>%{text}</b><br>' +
                         '✓ INCLUDED<br>' +
                         'Confidence: %{customdata[0]}<br>' +
                         '<extra></extra>'
        ))
    
    # Update layout for static
    fig_static.update_layout(
        title={
            'text': 't-SNE Visualization: Included Studies<br><sub>Included studies highlighted, others greyed out</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title='t-SNE Dimension 1',
        yaxis_title='t-SNE Dimension 2',
        height=800,
        width=1200,
        hovermode='closest',
        legend=dict(
            title='Study Status',
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1
        )
    )
    
    # Save static version
    fig_static.write_html(f"{output_dir}/tsne_visualization_included_only.html")
    fig_static.write_image(f"{output_dir}/tsne_visualization_included_only.pdf")
    logging.info(f"Static t-SNE (included only) saved to {output_dir}/")
    
    return fig_interactive, fig_static

In [78]:
def create_tsne_by_confidence(df_tsne, output_dir="outputs/visualizations"):
    """
    Create t-SNE visualization colored by confidence level (LLM-screened only).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Filter to LLM-screened only (exclude ML-screened)
    df_screened = df_tsne[df_tsne['ai_decision'] != 'screened_by_ml'].copy()
    
    if len(df_screened) == 0:
        logging.warning("No LLM-screened records for confidence visualization")
        return None
    
    # Color mapping
    color_map_conf = {
        'high': '#27ae60',
        'medium': '#f39c12',
        'low': '#e74c3c',
        'ml_filtered': '#95a5a6'
    }
    
    fig = go.Figure()
    
    for conf in ['high', 'medium', 'low']:
        df_subset = df_screened[df_screened['ai_confidence'] == conf]
        
        if len(df_subset) > 0:
            fig.add_trace(go.Scatter(
                x=df_subset['TSNE_1'],
                y=df_subset['TSNE_2'],
                mode='markers',
                name=f'{conf.title()} Confidence',
                marker=dict(
                    size=7,
                    color=color_map_conf[conf],
                    opacity=0.7,
                    line=dict(width=0.5, color='white')
                ),
                text=df_subset['Title'],
                customdata=df_subset['ai_decision'],
                hovertemplate='<b>%{text}</b><br>' +
                             'Confidence: ' + conf + '<br>' +
                             'Decision: %{customdata}<br>' +
                             'Method: LLM (GPT-5)<br>' +
                             '<extra></extra>'
            ))
    
    fig.update_layout(
        title={
            'text': 't-SNE Visualization by Confidence Level<br><sub>LLM-Screened Articles (GPT-5) Only</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        xaxis_title='t-SNE Dimension 1',
        yaxis_title='t-SNE Dimension 2',
        height=800,
        width=1200,
        hovermode='closest',
        legend=dict(
            title='Confidence',
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1
        )
    )
    
    fig.write_html(f"{output_dir}/tsne_by_confidence.html")
    logging.info(f"t-SNE by confidence saved to {output_dir}/")
    
    return fig

## 14. Generate Comprehensive PDF Report

In [79]:
def create_comprehensive_pdf_report(df, df_tsne=None, output_file="outputs/screening_report.pdf"):
    """
    Generate comprehensive PDF report with all visualizations using Matplotlib.
    """
    with PdfPages(output_file) as pdf:
        
        # Page 1: Title and Summary Statistics
        fig = plt.figure(figsize=(11, 8.5))
        fig.suptitle('AI-Powered Systematic Review Screening Report', fontsize=24, fontweight='bold', y=0.98)
        
        ax = fig.add_subplot(111)
        ax.axis('off')
        
        # Summary statistics
        n_total = len(df)
        n_include = (df['ai_decision'] == 'include').sum()
        n_exclude = (df['ai_decision'] == 'exclude').sum()
        n_uncertain = (df['ai_decision'] == 'uncertain').sum()
        
        summary_text = f"""
SCREENING SUMMARY (LLM - GPT-5)
{'='*80}

Total Records Screened by LLM:    {n_total:6d}
Records Included:                 {n_include:6d}  ({n_include/n_total*100:5.1f}%)
Records Excluded:                 {n_exclude:6d}  ({n_exclude/n_total*100:5.1f}%)
Records Uncertain:                {n_uncertain:6d}  ({n_uncertain/n_total*100:5.1f}%)

Note: Additional records were pre-filtered by ML (KNN) before LLM screening

CONFIDENCE DISTRIBUTION
{'='*80}
"""
        for conf in ['high', 'medium', 'low']:
            count = (df['ai_confidence'] == conf).sum()
            pct = count / n_total * 100 if n_total > 0 else 0
            summary_text += f"{conf.capitalize():12s}:               {count:6d}  ({pct:5.1f}%)\n"
        
        summary_text += f"\n\nCRITERIA MATCHING RATES\n{'='*80}\n"
        criteria = [
            ('Population', 'ai_population_match'),
            ('Exposure', 'ai_exposure_match'),
            ('Outcome', 'ai_outcome_match'),
            ('Study Design', 'ai_study_design_appropriate')
        ]
        for label, col in criteria:
            count = df[col].sum()
            pct = count / n_total * 100 if n_total > 0 else 0
            summary_text += f"{label:20s}:     {count:6d}  ({pct:5.1f}%)\n"
        
        summary_text += f"\n\nReport Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        
        ax.text(0.1, 0.95, summary_text, fontsize=11, verticalalignment='top',
                fontfamily='monospace', transform=ax.transAxes)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
        
        # Page 2: Decision Breakdown
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
        fig.suptitle('Decision Breakdown Analysis', fontsize=18, fontweight='bold')
        
        # Decision pie chart
        decision_counts = df['ai_decision'].value_counts()
        colors_decision = ['#27ae60', '#e74c3c', '#f39c12']
        axes[0, 0].pie(decision_counts.values, labels=decision_counts.index,
                       autopct='%1.1f%%', colors=colors_decision, startangle=90)
        axes[0, 0].set_title('Decision Distribution', fontweight='bold')
        
        # Confidence pie chart
        conf_counts = df['ai_confidence'].value_counts()
        colors_conf = ['#27ae60', '#f39c12', '#e74c3c']
        axes[0, 1].pie(conf_counts.values, labels=conf_counts.index,
                       autopct='%1.1f%%', colors=colors_conf, startangle=90)
        axes[0, 1].set_title('Confidence Levels', fontweight='bold')
        
        # Exclusion reasons
        df_excluded = df[df['ai_decision'] == 'exclude']
        if len(df_excluded) > 0:
            excl_counts = df_excluded['ai_exclusion_reason'].value_counts().head(8)
            axes[1, 0].barh(range(len(excl_counts)), excl_counts.values, color='#e74c3c')
            axes[1, 0].set_yticks(range(len(excl_counts)))
            axes[1, 0].set_yticklabels([x.replace('_', ' ').title() for x in excl_counts.index], fontsize=9)
            axes[1, 0].set_xlabel('Count')
            axes[1, 0].set_title('Top Exclusion Reasons', fontweight='bold')
            axes[1, 0].invert_yaxis()
        else:
            axes[1, 0].text(0.5, 0.5, 'No Exclusions', ha='center', va='center', fontsize=14)
            axes[1, 0].axis('off')
        
        # Criteria matching
        criteria_data = {
            'Population': df['ai_population_match'].sum(),
            'Exposure': df['ai_exposure_match'].sum(),
            'Outcome': df['ai_outcome_match'].sum(),
            'Study\nDesign': df['ai_study_design_appropriate'].sum()
        }
        axes[1, 1].bar(criteria_data.keys(), criteria_data.values(), color='#3498db')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Criteria Matching', fontweight='bold')
        axes[1, 1].tick_params(axis='x', rotation=0)
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
        
        # Page 3: Confidence Heatmap
        fig, ax = plt.subplots(figsize=(11, 8.5))
        fig.suptitle('Confidence Levels by Decision Type', fontsize=18, fontweight='bold')
        
        conf_matrix = pd.crosstab(df['ai_decision'], df['ai_confidence'])
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='RdYlGn', cbar_kws={'label': 'Count'},
                   ax=ax, linewidths=0.5, linecolor='gray')
        ax.set_xlabel('Confidence Level', fontweight='bold')
        ax.set_ylabel('Decision', fontweight='bold')
        
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
        
        # Page 4: t-SNE Visualization (if available)
        if df_tsne is not None and len(df_tsne) > 0:
            fig, ax = plt.subplots(figsize=(11, 8.5))
            fig.suptitle('t-SNE Visualization of Screening Results', fontsize=18, fontweight='bold')
            
            color_map = {
                'include': '#27ae60',
                'exclude': '#e74c3c',
                'uncertain': '#f39c12',
                'screened_by_ml': '#95a5a6'
            }
            
            # Show included studies prominently, others greyed out
            # First plot background (non-included)
            df_background = df_tsne[df_tsne['ai_decision'] != 'include']
            if len(df_background) > 0:
                ax.scatter(df_background['TSNE_1'], df_background['TSNE_2'],
                          c='#d3d3d3', label='Other Studies',
                          s=20, alpha=0.3, edgecolors='none')
            
            # Then plot included on top
            df_included = df_tsne[df_tsne['ai_decision'] == 'include']
            if len(df_included) > 0:
                ax.scatter(df_included['TSNE_1'], df_included['TSNE_2'],
                          c='#27ae60', label=f'Included (n={len(df_included)})',
                          s=50, alpha=0.8, edgecolors='white', linewidth=0.5)
            
            ax.set_xlabel('t-SNE Dimension 1', fontweight='bold')
            ax.set_ylabel('t-SNE Dimension 2', fontweight='bold')
            ax.legend(title='Decision', loc='best', framealpha=0.9)
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        
        # Metadata page
        d = pdf.infodict()
        d['Title'] = 'AI Screening Report'
        d['Author'] = 'GPT-5 Systematic Review Assistant'
        d['Subject'] = 'Automated screening results'
        d['Keywords'] = 'Systematic Review, AI Screening, GPT-5'
        d['CreationDate'] = datetime.now()
    
    logging.info(f"PDF report saved to {output_file}")
    print(f"✓ PDF report generated: {output_file}")

## 15. Run All Visualizations

In [80]:
# Create visualizations directory
os.makedirs("outputs/visualizations", exist_ok=True)

print("=" * 80)
print("GENERATING COMPREHENSIVE VISUALIZATIONS")
print("=" * 80)

# Load screening results
results_file = "outputs/ai_screening_full_results.csv"
if not os.path.exists(results_file):
    print(f"❌ Error: {results_file} not found. Please run screening first.")
else:
    df_results = pd.read_csv(results_file)
    print(f"\n✓ Loaded LLM screening results: {len(df_results)} records")
    
    # Load and merge with embeddings first (needed for pipeline diagram)
    embeddings_file = "outputs/csv_results_with_embeddings.csv"
    
    if os.path.exists(embeddings_file):
        print("\n[1/11] Loading and merging with embeddings...")
        df_merged = load_and_merge_with_embeddings(results_file, embeddings_file)
        
        # Debug: Show what we have
        print(f"   → Total records in merged data: {len(df_merged):,}")
        print(f"   → ML-excluded (screened_by_ml): {(df_merged['ai_decision'] == 'screened_by_ml').sum():,}")
        print(f"   → LLM-screened: {len(df_merged) - (df_merged['ai_decision'] == 'screened_by_ml').sum():,}")
        
        # 1. Screening Pipeline Diagram (ML + LLM)
        print("[2/11] Creating screening pipeline diagram...")
        fig_pipeline = create_screening_pipeline_diagram(df_merged)
        
        # 2. Funnel Plot (LLM only)
        print("[3/11] Creating LLM funnel plot...")
        fig_funnel = create_funnel_plot(df_results)
        
        # 3. Decision Breakdown
        print("[4/11] Creating decision breakdown plots...")
        fig_breakdown = create_decision_breakdown_plots(df_results)
        
        # 4. Exclusion Reasons Detail
        print("[5/11] Creating exclusion reasons detail...")
        fig_exclusion = create_exclusion_reasons_detail(df_results)
        
        # 5. Confidence Heatmap
        print("[6/11] Creating confidence heatmap...")
        fig_heatmap = create_confidence_heatmap(df_results)
        
        # 6. Prepare t-SNE data
        print("[7/11] Preparing t-SNE visualization...")
        df_tsne = prepare_tsne_data(df_merged, perplexity=50)
        
        # 7. Create t-SNE visualizations (returns tuple of interactive and static)
        print("[8/11] Creating t-SNE plots (interactive + included-only)...")
        fig_tsne_interactive, fig_tsne_static = create_tsne_visualization(df_tsne)
        
        # 8. t-SNE by confidence
        print("[9/11] Creating t-SNE by confidence...")
        fig_tsne_conf = create_tsne_by_confidence(df_tsne)
        
        # 9. Generate PDF Report
        print("[10/11] Generating comprehensive PDF report...")
        create_comprehensive_pdf_report(df_results, df_tsne, "outputs/screening_report.pdf")
        
        print("[11/11] Finalizing...")
        print("✓ All visualizations saved successfully!")
        
        print("\n" + "=" * 80)
        print("✓ ALL VISUALIZATIONS COMPLETE")
        print("=" * 80)
        print("\nGenerated files:")
        print("  📊 Pipeline & Summary:")
        print("     • outputs/visualizations/screening_pipeline.html (ML→LLM flow)")
        print("     • outputs/visualizations/screening_pipeline.pdf")
        print("     • outputs/visualizations/funnel_plot.html (LLM results)")
        print("     • outputs/visualizations/funnel_plot.pdf")
        print("\n  📈 Decision Analysis:")
        print("     • outputs/visualizations/decision_breakdown.html (4-panel dashboard)")
        print("     • outputs/visualizations/decision_breakdown.pdf")
        print("     • outputs/visualizations/exclusion_reasons_detail.html ⭐ (detailed exclusions)")
        print("     • outputs/visualizations/exclusion_reasons_detail.pdf")
        print("     • outputs/visualizations/confidence_heatmap.html")
        print("     • outputs/visualizations/confidence_heatmap.pdf")
        print("\n  🗺️  t-SNE Visualizations:")
        print("     • outputs/visualizations/tsne_visualization_interactive.html ⭐")
        print("       → Toggle: ML-excluded, LLM-excluded, Uncertain, Included")
        print("     • outputs/visualizations/tsne_visualization_included_only.html ⭐")
        print("       → Included studies highlighted, others greyed")
        print("     • outputs/visualizations/tsne_by_confidence.html (LLM confidence levels)")
        print("\n  📄 PDF Report:")
        print("     • outputs/screening_report.pdf (comprehensive summary)")
        print("\n" + "=" * 80)
        print("💡 INTERACTIVE FEATURES:")
        print("  • Pipeline diagram: Shows ML-excluded separately from LLM decisions")
        print("  • t-SNE interactive: Click legend to toggle any category on/off")
        print("  • Exclusion reasons: Hover for detailed counts and percentages")
        print("  • All categories visible: ML-excluded, LLM-excluded, Uncertain, Included")
        print("=" * 80)
    else:
        print(f"⚠️  Warning: {embeddings_file} not found.")
        print("Creating visualizations without t-SNE...")
        
        # Generate basic visualizations only
        print("\n[1/3] Creating funnel plot...")
        fig_funnel = create_funnel_plot(df_results)
        
        print("[2/3] Creating decision breakdown...")
        fig_breakdown = create_decision_breakdown_plots(df_results)
        
        print("[3/3] Generating PDF report...")
        create_comprehensive_pdf_report(df_results, None, "outputs/screening_report.pdf")
        
        print("\n✓ Basic visualizations complete (t-SNE requires embeddings file)")

2026-01-23 15:48:37,110 - INFO - Loaded 237 screening results


GENERATING COMPREHENSIVE VISUALIZATIONS

✓ Loaded LLM screening results: 237 records

[1/11] Loading and merging with embeddings...


2026-01-23 15:48:37,534 - INFO - Loaded 2920 records with embeddings
2026-01-23 15:48:37,551 - INFO - Merged dataset has 2920 records
2026-01-23 15:48:37,552 - INFO -   → Screened by LLM: 237
2026-01-23 15:48:37,552 - INFO -   → Screened by ML (KNN): 2683
2026-01-23 15:48:37,600 - INFO - Chromium init'ed with kwargs {}
2026-01-23 15:48:37,602 - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-01-23 15:48:37,603 - INFO - Temp directory created: /var/folders/l4/8t59gtdn11579mxq800nkfgw0000gn/T/tmpslzzw47y.
2026-01-23 15:48:37,604 - INFO - Opening browser.
2026-01-23 15:48:37,604 - INFO - Temp directory created: /var/folders/l4/8t59gtdn11579mxq800nkfgw0000gn/T/tmplgjwlg8c.
2026-01-23 15:48:37,605 - INFO - Temporary directory at: /var/folders/l4/8t59gtdn11579mxq800nkfgw0000gn/T/tmplgjwlg8c


   → Total records in merged data: 2,920
   → ML-excluded (screened_by_ml): 2,683
   → LLM-screened: 237
[2/11] Creating screening pipeline diagram...

   DEBUG: Total records in merged dataset: 2,920
   DEBUG: Records excluded by ML (KNN): 2,683
   DEBUG: Records screened by LLM (GPT-5): 237
   DEBUG: LLM decisions - Include: 46, Exclude: 185, Uncertain: 6
   DEBUG: All LLM exclusion reasons (6 total): {'wrong_exposure': 59, 'wrong_study_design': 58, 'wrong_outcome': 53, 'wrong_population': 9, 'animal_study': 3, 'no_original_data': 3}


2026-01-23 15:48:38,850 - INFO - Conforming 1 to file:///var/folders/l4/8t59gtdn11579mxq800nkfgw0000gn/T/tmpslzzw47y/index.html
2026-01-23 15:48:38,851 - INFO - Waiting on all navigates
2026-01-23 15:48:39,497 - INFO - All navigates done, putting them all in queue.
2026-01-23 15:48:39,498 - INFO - Getting tab from queue (has 1)
2026-01-23 15:48:39,498 - INFO - Got F7F2
2026-01-23 15:48:39,499 - INFO - Processing Screening_Pipeline.pdf
2026-01-23 15:48:39,499 - INFO - Sending big command for Screening_Pipeline.pdf.
2026-01-23 15:48:39,530 - INFO - Sent big command for Screening_Pipeline.pdf.
2026-01-23 15:48:39,597 - INFO - Reloading tab F7F2 before return.
2026-01-23 15:48:39,671 - INFO - Putting tab F7F2 back (queue size: 0).
2026-01-23 15:48:39,672 - INFO - Waiting for all cleanups to finish.
2026-01-23 15:48:39,672 - INFO - Exiting Kaleido
2026-01-23 15:48:39,672 - INFO - Closing browser.
2026-01-23 15:48:39,674 - INFO - Closing browser.
2026-01-23 15:48:39,675 - INFO - TemporaryDir

...

[8/11] Creating t-SNE plots (interactive + included-only)...


2026-01-23 15:48:55,577 - INFO - Conforming 1 to file:///var/folders/l4/8t59gtdn11579mxq800nkfgw0000gn/T/tmpeykql6u2/index.html
2026-01-23 15:48:55,578 - INFO - Waiting on all navigates
2026-01-23 15:48:56,808 - INFO - All navigates done, putting them all in queue.
2026-01-23 15:48:56,808 - INFO - Getting tab from queue (has 1)
2026-01-23 15:48:56,809 - INFO - Got 2D01
2026-01-23 15:48:56,809 - INFO - Processing t_SNE_Visualization_of_Screening_Results.pdf
2026-01-23 15:48:56,810 - INFO - Sending big command for t_SNE_Visualization_of_Screening_Results.pdf.
2026-01-23 15:48:56,906 - INFO - Sent big command for t_SNE_Visualization_of_Screening_Results.pdf.
2026-01-23 15:48:57,031 - INFO - Reloading tab 2D01 before return.
2026-01-23 15:48:57,093 - INFO - Putting tab 2D01 back (queue size: 0).
2026-01-23 15:48:57,094 - INFO - Waiting for all cleanups to finish.
2026-01-23 15:48:57,094 - INFO - Exiting Kaleido
2026-01-23 15:48:57,094 - INFO - Closing browser.
2026-01-23 15:48:57,097 - INF

[9/11] Creating t-SNE by confidence...
[10/11] Generating comprehensive PDF report...


2026-01-23 15:48:59,500 - INFO - PDF report saved to outputs/screening_report.pdf


✓ PDF report generated: outputs/screening_report.pdf
[11/11] Finalizing...
✓ All visualizations saved successfully!

✓ ALL VISUALIZATIONS COMPLETE

Generated files:
  📊 Pipeline & Summary:
     • outputs/visualizations/screening_pipeline.html (ML→LLM flow)
     • outputs/visualizations/screening_pipeline.pdf
     • outputs/visualizations/funnel_plot.html (LLM results)
     • outputs/visualizations/funnel_plot.pdf

  📈 Decision Analysis:
     • outputs/visualizations/decision_breakdown.html (4-panel dashboard)
     • outputs/visualizations/decision_breakdown.pdf
     • outputs/visualizations/exclusion_reasons_detail.html ⭐ (detailed exclusions)
     • outputs/visualizations/exclusion_reasons_detail.pdf
     • outputs/visualizations/confidence_heatmap.html
     • outputs/visualizations/confidence_heatmap.pdf

  🗺️  t-SNE Visualizations:
     • outputs/visualizations/tsne_visualization_interactive.html ⭐
       → Toggle: ML-excluded, LLM-excluded, Uncertain, Included
     • outputs/visual

## 16. View Summary Statistics

In [81]:
# Display comprehensive analysis with screening method breakdown
if os.path.exists("outputs/ai_screening_full_results.csv"):
    df_results = pd.read_csv("outputs/ai_screening_full_results.csv")
    
    # Also load merged data if available to show complete picture
    embeddings_file = "pubmed_results_with_embeddings.csv"
    if os.path.exists(embeddings_file):
        df_merged_stats = load_and_merge_with_embeddings(
            "outputs/ai_screening_full_results.csv",
            embeddings_file
        )
        
        print("\n" + "=" * 80)
        print("COMPLETE SCREENING PIPELINE SUMMARY")
        print("=" * 80)
        
        # Overall numbers
        n_total = len(df_merged_stats)
        n_ml_screened = (df_merged_stats['ai_decision'] == 'screened_by_ml').sum()
        n_llm_screened = n_total - n_ml_screened
        
        print(f"\n📚 Total Records: {n_total:,}")
        print(f"   • Screened by ML (KNN):  {n_ml_screened:5d} ({n_ml_screened/n_total*100:5.1f}%)")
        print(f"   • Screened by LLM (GPT-5): {n_llm_screened:5d} ({n_llm_screened/n_total*100:5.1f}%)")
    
    print("\n" + "=" * 80)
    print("LLM SCREENING RESULTS (GPT-5)")
    print("=" * 80)

    # Decision distribution
    print("\n📊 DECISION DISTRIBUTION:")
    decision_counts = df_results['ai_decision'].value_counts()
    for decision, count in decision_counts.items():
        pct = count / len(df_results) * 100
        print(f"  {decision:12s}: {count:5d} ({pct:5.1f}%)")

    # Confidence levels
    print("\n🎯 CONFIDENCE LEVELS:")
    conf_counts = df_results['ai_confidence'].value_counts()
    for conf, count in conf_counts.items():
        pct = count / len(df_results) * 100
        print(f"  {conf:12s}: {count:5d} ({pct:5.1f}%)")

    # Criteria matching
    print("\n✅ CRITERIA MATCHING:")
    criteria = [
        ('Population', 'ai_population_match'),
        ('Exposure', 'ai_exposure_match'),
        ('Outcome', 'ai_outcome_match'),
        ('Study Design', 'ai_study_design_appropriate')
    ]
    for label, col in criteria:
        count = df_results[col].sum()
        pct = count / len(df_results) * 100
        print(f"  {label:20s}: {count:5d} ({pct:5.1f}%)")

    # Confidence by decision
    print("\n📉 CONFIDENCE BY DECISION:")
    conf_by_decision = pd.crosstab(df_results['ai_decision'], df_results['ai_confidence'])
    print(conf_by_decision)

    # Exclusion reasons
    df_excluded = df_results[df_results['ai_decision'] == 'exclude']
    if len(df_excluded) > 0:
        print("\n🚫 TOP EXCLUSION REASONS:")
        excl_counts = df_excluded['ai_exclusion_reason'].value_counts().head(10)
        for reason, count in excl_counts.items():
            pct = count / len(df_excluded) * 100
            print(f"  {reason:40s}: {count:4d} ({pct:5.1f}%)")
else:
    print("No results file found. Run screening first.")


LLM SCREENING RESULTS (GPT-5)

📊 DECISION DISTRIBUTION:
  exclude     :   185 ( 78.1%)
  include     :    46 ( 19.4%)
  uncertain   :     6 (  2.5%)

🎯 CONFIDENCE LEVELS:
  high        :   212 ( 89.5%)
  medium      :    19 (  8.0%)
  low         :     6 (  2.5%)

✅ CRITERIA MATCHING:
  Population          :   197 ( 83.1%)
  Exposure            :   120 ( 50.6%)
  Outcome             :   122 ( 51.5%)
  Study Design        :   156 ( 65.8%)

📉 CONFIDENCE BY DECISION:
ai_confidence  high  low  medium
ai_decision                     
exclude         182    0       3
include          30    0      16
uncertain         0    6       0

🚫 TOP EXCLUSION REASONS:
  wrong_exposure                          :   59 ( 31.9%)
  wrong_study_design                      :   58 ( 31.4%)
  wrong_outcome                           :   53 ( 28.6%)
  wrong_population                        :    9 (  4.9%)
  animal_study                            :    3 (  1.6%)
  no_original_data                        :    3 

## 17. Sample Decision Examples

In [18]:
# Display sample of each decision category
if os.path.exists("outputs/ai_screening_full_results.csv"):
    df = pd.read_csv("outputs/ai_screening_full_results.csv")

    for decision in ['include', 'exclude', 'uncertain']:
        df_subset = df[df['ai_decision'] == decision]
        if len(df_subset) > 0:
            print("\n" + "="*80)
            print(f"Sample {decision.upper()} decision:")
            print("="*80)
            sample = df_subset.iloc[0]
            print(f"PMID: {sample['PMID']}")
            print(f"Title: {sample['Title']}")
            print(f"\nDecision: {sample['ai_decision']}")
            print(f"Confidence: {sample['ai_confidence']}")
            print(f"Exclusion reason: {sample['ai_exclusion_reason']}")
            print(f"\nReasoning: {sample['ai_reasoning']}")
            print(f"\nCriteria matches:")
            print(f"  - Population: {sample['ai_population_match']}")
            print(f"  - Exposure: {sample['ai_exposure_match']}")
            print(f"  - Outcome: {sample['ai_outcome_match']}")
            print(f"  - Study design: {sample['ai_study_design_appropriate']}")
else:
    print("No results file found.")


Sample INCLUDE decision:
PMID: 25163522
Title: Climate change, crop production and child under nutrition in Ethiopia; a longitudinal panel study.

Decision: include
Confidence: high
Exclusion reason: nan

Reasoning: Human child population in Ethiopia with exposure to temperature (and rainfall) and outcome of child undernutrition. The study appears longitudinal/panel, assessing weather effects on growth outcomes. Meets inclusion criteria for population, exposure, outcome, and study design.

Criteria matches:
  - Population: True
  - Exposure: True
  - Outcome: True
  - Study design: True

Sample EXCLUDE decision:
PMID: 1161182
Title: [Environmental factors in juvenile delinquency].

Decision: exclude
Confidence: high
Exclusion reason: wrong_exposure

Reasoning: The article discusses environmental and sociocultural factors in juvenile delinquency without any heat or temperature-related exposure. It also appears to be a conceptual/commentary piece rather than original empirical research.

## 18. Generate PRISMA Flow Diagram Data

In [29]:
def generate_prisma_data(results_file: str = "outputs/ai_screening_full_results.csv"):
    """
    Generate data for PRISMA 2020 flow diagram.
    """
    df = pd.read_csv(results_file)
    
    prisma_data = {
        "identification": {
            "records_identified_databases": len(df),
            "records_removed_before_screening": 0  # If you have deduplication
        },
        "screening": {
            "records_screened": len(df),
            "records_excluded": (df['ai_decision'] == 'exclude').sum(),
            "records_uncertain": (df['ai_decision'] == 'uncertain').sum()
        },
        "included": {
            "records_included_for_fulltext": (df['ai_decision'] == 'include').sum()
        },
        "exclusion_reasons": df[df['ai_decision'] == 'exclude']['ai_exclusion_reason'].value_counts().to_dict()
    }
    
    # Save as JSON
    output_file = results_file.replace('.csv', '_prisma.json')
    with open(output_file, 'w') as f:
        json.dump(prisma_data, f, indent=2)
    
    print("PRISMA Flow Diagram Data:")
    print(json.dumps(prisma_data, indent=2))
    print(f"\nSaved to: {output_file}")
    
    return prisma_data

# Generate PRISMA data
if os.path.exists("outputs/ai_screening_full_results.csv"):
    prisma_data = generate_prisma_data("outputs/ai_screening_full_results.csv")
else:
    print("No results file found.")

TypeError: Object of type int64 is not JSON serializable

## 19. Summary and Next Steps

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════════════════
AI-POWERED SYSTEMATIC REVIEW SCREENING - WORKFLOW SUMMARY
═══════════════════════════════════════════════════════════════════════════════

✓ Latest OpenAI SDK (>=1.0.0) with native structured outputs
✓ GPT-5 reasoning models with adjustable reasoning_effort
✓ Pydantic BaseModel for type-safe, validated responses
✓ PRISMA-aligned exclusion reasons
✓ Comprehensive visualizations (funnel plots, t-SNE, heatmaps)
✓ Interactive HTML outputs and PDF reports
✓ Incremental progress saving with resume capability

RECOMMENDED WORKFLOW:

1. SETUP ✓
   - OpenAI API key configured
   - Inclusion criteria loaded from text file
   - PubMed data with embeddings ready
   - Visualization packages installed (plotly, matplotlib, seaborn, kaleido)

2. TEST SCREENING (Sections 8-9):
   - Test on 10-20 abstracts first
   - Review decision quality and reasoning
   - Adjust model/reasoning_effort if needed

3. FULL SCREENING (Section 10):
   - Run on entire dataset with progress tracking
   - Monitor API costs and rate limits
   - Results saved incrementally to outputs/

4. ANALYSIS & VISUALIZATION (Sections 11-15):
   - Load and merge results with embeddings
   - Generate comprehensive statistics
   - Create interactive visualizations:
     * PRISMA funnel plots
     * Decision breakdown charts
     * Confidence heatmaps
     * t-SNE clustering with filtering
   - Export PDF report

5. REVIEW & VALIDATION (Sections 16-17):
   - Review summary statistics
   - Examine sample decisions
   - Identify uncertain cases for manual review
   - Calculate validation metrics if ground truth available

6. EXPORT & REPORTING (Section 18):
   - Generate PRISMA flow diagram data
   - Export for manuscript preparation
   - Document AI screening methodology

VISUALIZATION OUTPUTS:
  📊 Pipeline & Flow:
    • outputs/visualizations/screening_pipeline.html ⭐
      → Sankey diagram showing ML-excluded separately from LLM decisions
      → Clear flow: Total → ML-excluded (grey) + LLM → Excluded/Uncertain/Included
    • outputs/visualizations/funnel_plot.html (LLM screening funnel)
  
  📈 Decision Analysis:
    • outputs/visualizations/decision_breakdown.html (4-panel dashboard)
    • outputs/visualizations/exclusion_reasons_detail.html ⭐
      → Detailed LLM exclusion reasons with counts and percentages
    • outputs/visualizations/confidence_heatmap.html (decision × confidence)
  
  🗺️  t-SNE Visualizations:
    • outputs/visualizations/tsne_visualization_interactive.html ⭐⭐
      → Toggle ALL categories: ML-excluded, LLM-excluded, Uncertain, Included
      → Click legend items to show/hide any category
      → Different marker sizes for better visibility
    • outputs/visualizations/tsne_visualization_included_only.html ⭐
      → Included studies highlighted in green, others greyed out
    • outputs/visualizations/tsne_by_confidence.html (LLM confidence clustering)
  
  📄 Reports:
    • outputs/screening_report.pdf (comprehensive multi-page report)

SCREENING CATEGORIES (ALL VISIBLE):
  • "Excluded by ML (KNN)": Grey markers, smallest size - pre-filtered records
  • "Excluded by LLM": Red markers - with detailed exclusion reasons
  • "Uncertain (LLM)": Orange markers - requires manual review
  • "Included (LLM)": Green markers, largest size - final included studies
  
INTERACTIVE FEATURES:
  • Pipeline: Sankey diagram with proper branching showing ML vs LLM paths
  • t-SNE: Click any legend item to toggle category visibility
  • Exclusions: Hover for detailed breakdown of why studies were excluded
  • All visualizations clearly distinguish ML pre-filtering from LLM decisions

GPT-5 MODEL RECOMMENDATIONS:
  • Testing: gpt-5-mini with medium reasoning (~$8-15 per 1K abstracts)
  • Production: gpt-5 with medium reasoning (~$15-30 per 1K abstracts)
  • Complex cases: gpt-5 with high reasoning (~$30-60 per 1K abstracts)
  • Bulk screening: gpt-5-nano with low reasoning (~$3-6 per 1K abstracts)

REASONING EFFORT GUIDE:
  • minimal: Fast, shallow analysis (simple decisions)
  • low: Light reasoning (straightforward cases)
  • medium: Balanced depth vs speed (RECOMMENDED DEFAULT)
  • high: Deep multistep reasoning (complex/uncertain cases)

QUALITY ASSURANCE TIPS:
  ✓ Validate on subset with manual screening (calculate kappa)
  ✓ Review all low-confidence decisions
  ✓ Examine exclusion reason distributions for anomalies
  ✓ Use t-SNE clusters to identify potential misclassifications
  ✓ Document methodology for manuscript methods section

NEXT STEPS:
  1. Run full screening if not yet completed
  2. Generate all visualizations (Section 15)
  3. Review uncertain cases manually
  4. Prepare PRISMA flow diagram (Section 18)
  5. Document validation metrics

═══════════════════════════════════════════════════════════════════════════════
""")

## Optional: Compare Human and AI Screening Results

This cell compares the human screening results (from the `human_screening` folder) and the AI screening results (from `ai_screening_full_results.csv`) by matching on the `Title` column. It reports sensitivity, specificity, precision, recall, and F1-score for the AI screening, using both direct and fuzzy matching (with high confidence).

In [83]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import os

# Optional cell: Compare Human and AI Screening Results by Title
# Set these paths as needed
human_folder = "../scripts/outputs/human_screening/"
human_files = [f for f in os.listdir(human_folder) if f.endswith(".csv")]
if not human_files:
    print("No human screening CSV found in outputs/human_screening/")
else:
    human_file = os.path.join(human_folder, human_files[0])
    ai_file = "../scripts/outputs/ai_screening_full_results.csv"
    if not os.path.exists(ai_file):
        print(f"AI screening file not found: {ai_file}")
    else:
        df_human = pd.read_csv(human_file)
        df_ai = pd.read_csv(ai_file)
        # Standardize column names
        df_human.columns = [c.strip() for c in df_human.columns]
        df_ai.columns = [c.strip() for c in df_ai.columns]

        # Resolve column names (case-insensitive)
        human_title_col = next((c for c in df_human.columns if c.lower() == "title"), None)
        ai_title_col = next((c for c in df_ai.columns if c.lower() == "title"), None)
        ai_decision_col = next((c for c in df_ai.columns if c.lower() == "ai_decision"), None)

        if not human_title_col:
            raise KeyError("Human CSV missing a Title column")
        if not ai_title_col:
            raise KeyError("AI CSV missing a Title column")
        if not ai_decision_col:
            raise KeyError("AI CSV missing an ai_decision column")

        # Human: inclusion is presence in the file
        human_titles = df_human[human_title_col].astype(str).str.strip()
        human_included = set(human_titles)

        # AI: inclusion is ai_decision == 'include'
        ai_titles = df_ai[ai_title_col].astype(str).str.strip()
        ai_included = set(
            df_ai[
                df_ai[ai_decision_col]
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["include", "uncertain"])
            ][ai_title_col]
            .astype(str)
            .str.strip()
        )

        all_titles = list(set(human_titles) | set(ai_titles))
        y_true = [1 if t in human_included else 0 for t in all_titles]
        y_pred = [1 if t in ai_included else 0 for t in all_titles]

        def print_plain_language(y_true_list, y_pred_list, label):
            tn, fp, fn, tp = confusion_matrix(y_true_list, y_pred_list, labels=[0, 1]).ravel()
            included_total = tp + fn
            excluded_total = tn + fp
            sensitivity = tp / included_total if included_total else float("nan")
            specificity = tn / excluded_total if excluded_total else float("nan")
            print(f"{label} Plain-Language Interpretation:")
            print(
                f"- Out of {included_total} human-included records, the AI included {tp} "
                f"and missed {fn} (sensitivity/recall = {sensitivity:.3f})."
            )
            print(
                f"- Out of {excluded_total} human-excluded records, the AI excluded {tn} "
                f"but incorrectly included {fp} (specificity = {specificity:.3f})."
            )
            print("- Included is the positive class; excluded is the negative class.")

        print("Direct Title Match Results:")
        print(classification_report(y_true, y_pred, target_names=["Excluded", "Included"]))
        print_plain_language(y_true, y_pred, "Direct Match")

        # Fuzzy match (high confidence)
        ai_included_fuzzy = set()
        for t in ai_included:
            match, score, *_ = process.extractOne(t, human_titles, scorer=fuzz.token_sort_ratio)
            if score >= 95:
                ai_included_fuzzy.add(match)
        y_pred_fuzzy = [1 if t in ai_included_fuzzy else 0 for t in all_titles]
        print("Fuzzy Title Match Results (score >= 95):")
        print(classification_report(y_true, y_pred_fuzzy, target_names=["Excluded", "Included"]))
        print_plain_language(y_true, y_pred_fuzzy, "Fuzzy Match")


Direct Title Match Results:
              precision    recall  f1-score   support

    Excluded       0.80      0.91      0.85       184
    Included       0.67      0.46      0.55        76

    accuracy                           0.78       260
   macro avg       0.74      0.68      0.70       260
weighted avg       0.76      0.78      0.76       260

Direct Match Plain-Language Interpretation:
- Out of 76 human-included records, the AI included 35 and missed 41 (sensitivity/recall = 0.461).
- Out of 184 human-excluded records, the AI excluded 167 but incorrectly included 17 (specificity = 0.908).
- Included is the positive class; excluded is the negative class.
Fuzzy Title Match Results (score >= 95):
              precision    recall  f1-score   support

    Excluded       0.84      1.00      0.92       184
    Included       1.00      0.55      0.71        76

    accuracy                           0.87       260
   macro avg       0.92      0.78      0.81       260
weighted avg   